# 2. Target Leakage Investigation

An early version of this project reported **ROC-AUC 1.0000** on a held-out test set. Logistic regression, random forest and XGBoost all scored exactly 1.0.

This notebook is how that was tracked down. The original notebook is kept unmodified at `archive/original_EDA_and_Modeling_LEAKING.ipynb` as evidence.

## The symptom

Three different model families, all perfect, on 29,734 held-out rows:

```
                     Accuracy  Precision  Recall  F1 Score  ROC-AUC
Logistic Regression       1.0        1.0     1.0       1.0      1.0
Random Forest             1.0        1.0     1.0       1.0      1.0
XGBoost                   1.0        1.0     1.0       1.0      1.0
```

Logistic regression reaching 1.0 is the tell. A linear model cannot perfectly separate a real credit population. When a linear model and a tree ensemble agree exactly, the cause is the data, not the model.

In [1]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 110)
pd.set_option("display.max_columns", 40)

DATA = "../data/Loan_Default.csv"

df = pd.read_csv(DATA)
y = df['Status']

## Hypothesis: a feature is a copy of the target

The original feature engineering created `_missing` indicator columns. If a column is only ever populated for non-defaulted loans, that indicator *is* the label.

In [2]:
rows = []
for c in df.columns:
    if c == "Status" or not df[c].isna().any():
        continue
    miss = df[c].isna()
    rows.append({
        "column": c,
        "pct_missing": 100 * miss.mean(),
        "P(default|missing)": y[miss].mean(),
        "P(default|present)": y[~miss].mean(),
        "agreement_with_target": (miss.astype(int) == y).mean(),
    })
pd.DataFrame(rows).sort_values("agreement_with_target", ascending=False).round(4)

,column,pct_missing,P(default|missing),P(default|present),agreement_with_target
4,Interest_rate_spread,24.6445,1.0000,0.0000,1.0000
3,rate_of_interest,24.5100,1.0000,0.0018,0.9987
5,Upfront_charges,26.6644,0.9204,0.0014,0.9777
8,property_value,10.1554,0.9999,0.1613,0.8551
12,LTV,10.1554,0.9999,0.1613,0.8551
13,dtir1,16.2245,0.6762,0.1632,0.8107
10,age,0.1345,1.0000,0.2454,0.7549
11,submission_of_application,0.1345,1.0000,0.2454,0.7549
6,term,0.0276,0.3659,0.2464,0.7535
7,Neg_ammortization,0.0814,0.2645,0.2464,0.7532


`Interest_rate_spread` agrees with the target on **100.000%** of rows. Not 99.9% — every single one of the 148,670.

In [3]:
agreement = (df["Interest_rate_spread"].isna().astype(int) == y).mean()
print(f"agreement: {agreement:.6%}  ({(df['Interest_rate_spread'].isna().astype(int) == y).sum():,} / {len(y):,})")
print()
print(pd.crosstab(df["rate_of_interest"].isna(), df["Status"]))

agreement: 100.000000%  (148,670 / 148,670)

Status                 0      1
rate_of_interest               
False             112031    200
True                   0  36439


## Why it happens

`rate_of_interest`, `Interest_rate_spread` and `Upfront_charges` are loan **pricing** fields. They only exist for loans that were originated and priced. Their presence is a consequence of the outcome being predicted, not information available when the decision is made.

This is temporal leakage: post-decision information leaking backwards.

## What it did to the model

Feature importances from the original trained artifact:

```
0.49691  rate_of_interest_missing
0.47780  Interest_rate_spread_missing
0.01121  age_nan
0.00857  credit_type_EQUI
0.00552  Upfront_charges_missing
0.00000  Credit_Score
0.00000  LTV
0.00000  dtir1
0.00000  income
```

97.5% of importance on two leakage indicators. 82 of 87 features at exactly zero, including every genuine underwriting variable. It was a lookup table, not a credit model.

## The second layer

Dropping the three pricing columns is not enough. `LTV`, `property_value` and `dtir1` leak through missingness too — more weakly, but enough to matter.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

M = pd.DataFrame({c: df[c].isna().astype(int)
                  for c in ["property_value", "LTV", "dtir1"]})
Xtr, Xte, ytr, yte = train_test_split(M, y, test_size=.2, stratify=y, random_state=42)
m = XGBClassifier(n_estimators=50, max_depth=3, eval_metric="logloss",
                  random_state=42).fit(Xtr, ytr)
print(f"AUC using ONLY those three missingness flags: {roc_auc_score(yte, m.predict_proba(Xte)[:,1]):.4f}")

AUC using ONLY those three missingness flags: 0.7155


0.7155 from nothing but three yes/no flags. And median imputation does not fix it: every imputed row lands on the same exact value, which a tree isolates as easily as a NaN.

The fix adopted is **complete-case training** — drop rows missing those columns rather than impute them. It costs 16.2% of the data and closes the channel completely.

## Performance at each stage of the fix

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import average_precision_score

LEAK = ["rate_of_interest", "Interest_rate_spread", "Upfront_charges"]

def run(label, drop, complete_case=False, drop_protected=False):
    d = df.dropna(subset=["property_value","LTV","dtir1"]) if complete_case else df
    yy = d["Status"]
    cols = ["Status"] + drop + (["Gender","age"] if drop_protected else [])
    X = d.drop(columns=[c for c in cols if c in d.columns])
    num = X.select_dtypes(include=np.number).columns.tolist()
    cat = [c for c in X.columns if c not in num]
    pre = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num),
        ("cat", Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                          ("o", OneHotEncoder(handle_unknown="ignore"))]), cat)])
    Xtr, Xte, ytr, yte = train_test_split(X, yy, test_size=.2, stratify=yy, random_state=42)
    p = Pipeline([("pre", pre), ("m", XGBClassifier(
        n_estimators=250, learning_rate=.08, max_depth=6, subsample=.8,
        colsample_bytree=.8, min_child_weight=4, eval_metric="logloss",
        random_state=42, n_jobs=-1))]).fit(Xtr, ytr)
    pr = p.predict_proba(Xte)[:, 1]
    return {"configuration": label, "n": len(d), "ROC-AUC": roc_auc_score(yte, pr),
            "PR-AUC": average_precision_score(yte, pr)}

results = [
    run("original (all leakage present)", []),
    run("drop ID only", ["ID"]),
    run("drop pricing columns + ID", LEAK + ["ID"]),
    run("complete cases, no protected attrs", LEAK + ["ID"], True, True),
]
pd.DataFrame(results).round(4)

,configuration,n,ROC-AUC,PR-AUC
0,original (all leakage present),148670,1.0000,1.0000
1,drop ID only,148670,1.0000,1.0000
2,drop pricing columns + ID,148670,0.8980,0.8482
3,"complete cases, no protected attrs",124547,0.8269,0.6331


From a fake 1.0000 to an honest **0.8244**. For a retail PD model that is a good result — production scorecards commonly land in this range.

## Where the real signal turned out to be

With the leakage gone, the drivers are interpretable: balloon payments, negative amortisation, business purpose, application channel. Loan structure, not a NaN pattern.

## What now prevents this recurring

| Control | Where |
|---|---|
| Training aborts if test AUC > 0.95 | `LeakageGuardError` in `models/train.py` |
| Missingness indicators forbidden | `test_feature_engineering_creates_no_missingness_indicators` |
| Excluded columns cannot reach features | `test_excluded_columns_absent_from_loaded_features` |
| No feature may hold >50% of SHAP importance | `test_no_single_feature_dominates` |
| Predictions must not pile up at 0 and 1 | `test_predictions_are_not_degenerate` |
| Data profiler flags leaky missingness | `data/quality.py` |

The guardrail is the important one. A model that scores too well now **fails the build**, which is exactly the control that was missing when 1.0000 was accepted as a result.

## What I took from this

1. A perfect score is a bug report, not an achievement.
2. Logistic regression makes a good leak detector — if it matches a gradient-boosted ensemble at a suspiciously high score, look at the data.
3. Check feature importance before believing a metric. Two features holding 97.5% was visible immediately.
4. Missingness indicators are dangerous on observational data. They are fine when missingness is genuinely random; here it *was* the outcome.
5. Leakage has layers — the obvious fix left a 0.7155 channel open.
6. Encode the finding as a test. Documentation is forgotten; a failing build is not.